# Assignment 2

This is the second assignment for the `GEOML` course, and the focus is on **regression**.

In this assignment, you will work with the training set from the [2016 SEG Machine Learning Contest](https://github.com/seg/2016-ml-contest).

This data was used in one of the past labs, but here it was modified and the *missing data* is not the same.

## Checking the data

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import *

plt.style.use("seaborn")

The provided data is a CSV file with well logs information from different wells. There are 5 well logs and 2 indicators:

* Gamma ray (GR)
* Resistivity (ILD\_log10)
* Photoelectric effect (PE)
* Neutron-density porosity difference (DeltaPHI)
* Average neutron-density porosity (PHIND)
* Nonmarine/marine indicator (NM\_M)
* Relative position (RELPOS)

The goal is to train a model able to classify 9 different types of rocks, as listed in the following table:

| Facies | Description | Label | Adjacent Facies|
| :---:  |    :---:    | :---: |      :---:     |
|   1    | Nonmarine Sandstone | SS | 2 |
|   2    | Nonmarine coarse siltstone | CSiS | 1,3 |
|   3    | Nonmarine fine siltstone | FSiS | 2 |
|   4    | Marine siltstone and shale | SiSh | 5 |
|   5    | Mudstone | MS | 4,6 |
|   6    | Wackestone | WS | 5,7,8 |
|   7    | Dolomite | D | 6,8 |
|   8    | Packstone-grainstone | PS | 6,7,9 |
|   9    | Phylloid-algal bafflestone | BS | 7,8 |

However the lithofacies are included in the data, they MUST NOT BE USED FOR THE REGRESSION.

Loading the CSV file to *pandas* dataframe and cheking some of the features:

In [ ]:
data = pd.read_csv("./data/data.csv")
data.head()

The data was modified by including `NANs` to **one or more logs**.

In this assignment, you will analyze the data and answer questions along the way and, in the end, perform data imputation using a regression algorithm (of your choice).

### Part 1: How many missing or wrong (NANs) data values are there for each column (in absolute numbers)?

In [ ]:
# YOUR CODE HERE


## Plotting the data

The file `utils.py` contains a function to plot the data. Let's plot the logs of the well `SHRIMPLIN`:

In [ ]:
well = "SHRIMPLIN"

make_facies_log_plot(data[data['Well Name'] == well])

### Part 2: Plot the logs for each the well! 

P.S.: remember that the matplotlib can be unfriendly to `NANs` in the data, so try to replace them with 0s or any other way of your choice. Just don't do a regression yet.

In [ ]:
# YOUR CODE HERE


## Data Imputation

Now things are getting a little more complicated than in the lecture, as more than one log has missing data, and **not the whole well for all times**. \
In this part we will do data imputation (interpolation) with regression algorithms.

As a best practice, we suggest to do the data imputation first for the log with the less number of missing data values (only using complete logs).\
After a well is completed, it can be included as a data.

### Part 3: Using the method of your choice, replace `only the missing data` for each log that contains NAN's with the regressions outputs, using the other logs as features. Use only the 'logs' as features, excluding the indicators, formation, and facies. You can use the `Depth` if you like.

This is a more broad question, with an open solution. You can choose any type of regression model, but we recommend non-linear ones. And you can also choose which package (sklearn, tensorflow, etc) you want to use. Just make sure to comment and explain each step of your solution.

As a best practice, we suggest to do the data imputation first for the log with the less number of missing data first (only using complete logs), and then the next ones including the newest complete one as feature.

Remember to always separate one well for validation, and select an appropriate metric (or metrics) to evaluate your solution.

In [ ]:
# YOUR CODE HERE


Remember to retrain your model with the validation set included to the training set before predict the missing logs.

## Challenge: build your own linear regression function

We used several libraries for regression, classification, and clustering, but can we create our own functions?

Many of them are complicated and are results of full PhD thesis, but some are quite trivial and simple. 

So, for this challenge, can you build your own linear regression function? Let's do it step-by-step.

First, let's load important packages:

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from math import ceil
import matplotlib.pyplot as plt
plt.style.use('seaborn')

For this notebook, we are going to use an open source dataset of features of houses in Boston to predict its price.

So, let's read the csv data and convert it to a pandas DataFrame.

In [ ]:
data = pd.read_csv('./data/BostonHousing.csv')
data

Let's see the definition of each columns:

| Column Name | Description |
| :-----: |    :---:    |
|   id    | Identification number of the house |
|   date    | Date of the aquired information |
|   price    | Price of the house |
|   bedrooms    | Number of bedrooms in the house |
|   bathrooms    | Number of bathrooms in the houne |
|   sqft_living    | Square footage of the living area |
|   sqft_lot    | Square footage of the lot area |
|   floors    | Number of floors in the house |
|   waterfront    | If it has water front |
|   view    | If it has view |
|   condition    | Condition of the house |
|   grade    | Grade given to the house |
|   sqft_above    | Square footage of the up part of the house |
|   sqft_basement    | Square footage of the basement |
|   yr_built    | Year the house was built |
|   yr_renovated    | Year of the last renovation of the house |
|   zipcode    | Zipcode |
|   lat    | Latitude of the house |
|   long    | Longitude of the house |
|   sqft_living15    | Square footage of the living area of the 15 nearest houses |
|   sqft_lot15    | Square footage of the lot area of the 15 nearest houses |


Let's take a look on the data:

In [ ]:
data.info()

In [ ]:
round(data.describe(), 2)

### Goal: predict the price of the houses using the house information.

First, let's just split the data into train and validation sets. As the rows are independent of each other, we can do just a random split.

Use the [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) function for that, with $20\%$ of the the data as validation:

In [ ]:
# YOUR CODE HERE


### Linear Regression with Single Feature

We want to fit a line that best estimates all the values in the training set. For that, we need to find 2 parameters: the slope and the intercept. The equation of the line is:

$$y_{i}^{,}(x_i) = w_0 + w_{1}*x_i$$

Now, we want to find the parameters $w_0$ and $w_1$ that reduces the cost function (the sum of the squared difference between measured data $y_i$ to the predicted data $y_{i}^{,}$):

$$RSS(w_0,w_1) = \sum_{i=1}^{N}(y_i - y_{i}^{,})^2 = \sum_{i=1}^{N}(y_i - [w_0 + w_{1}*x_i])^2$$

Minimizing the cost function means to take the derivative of cost function for each parameter ($w_0$ and $w_1$) and make it equal to zero. This leads to 2 simple formulas for $w_0$ and $w_1$:

$$w_0 = \frac{\sum_{i=1}^{N}y_i}{N} - w_{1}\frac{\sum_{i=1}^{N}x_i}{N}$$

and

$$w_1 = \frac{\sum_{i=1}^{N}y_{i}x_{i} - \frac{\sum_{i=1}^{N}y_{i}\sum_{i=1}^{N}x_{i}}{N}}{\sum_{i=1}^{N}x_{i}^2 - \frac{\sum_{i=1}^{N}x_{i}\sum_{i=1}^{N}x_{i}}{N}}$$

With the equations above, it is possible to compute the intercept ($w_0$) and the slope ($w_1$) that best predict the output $(y_{i}^{'})$ given the input $x_i$ and the measured data $y_i$ (for one feature only).

Now, let's create the function that gets the input feature $x_i$ and the measured data $y_i$ of the training set, and return the intercept $w_0$ and the slope $w_1$.

Note: you need to complete the code at some places.

In [ ]:
def linear_regression_single(input_feature, measured_data):
    # First, let's compute the sums and squared sums of the parameters equations
    Isum = input_feature.sum()
    Msum = measured_data.sum()
    IMsum = sum([input_feature[i]*measured_data[i] for i in range(len(input_feature))])
    IIsum = sum([input_feature[i]*input_feature[i] for i in range(len(input_feature))])

    # We need to compute the slope first
    num = IMsum-(1./len(input_feature)*(Isum*Msum))
    den = IIsum-(1./len(input_feature)*(Isum*Isum))
    # YOUR CODE HERE #
    slope = 
    
    # Now that we have the slope, we can compute the intercept
    intercept = (1./len(input_feature))*Msum-slope*Isum*(1./len(input_feature))
    
    # Return the parameters
    return intercept, slope

Now, let's test our function. For that, let's use the `sqrt_living` as feature:

In [ ]:
intercept_1, slope_1 = linear_regression_single(train['sqft_living'].values, train['price'].values)

print("Intercept: " + str(intercept_1))
print("Slope: " + str(slope_1))

The next step is to calculate the estimations $y_{i}^{,}$ by using the linear equation:

$$y_{i}^{,}(x_i) = w_0 + w_{1}*x_i$$

Let's create a function that gets the input feature $x_i$ and the measured data $y_i$. It needs to estimate the intercept $w_0$ and slope $w_1$ and calculate the predictions $y_{i}^{,}$ using the linear equation above.

In [ ]:
def regression_predictions_single(input_feature, measured_data):
    # First, we need to estimete the intercept and the slope
    intercept, slope = linear_regression_single(input_feature, measured_data)
    
    # Now, compute the predictions
    # YOUR CODE HERE #
    predicted_values = 
    
    # Return outputs
    return predicted_values, intercept, slope

Let's test our function:

In [ ]:
predictions, intercept, slope = regression_predictions_single(train['sqft_living'].values, train['price'].values)

print("Intercept: " + str(intercept))
print("Slope: " + str(slope))

In [ ]:
plt.figure(figsize = (10,4))
plt.plot(train['sqft_living'], train['price']/1000000,'.', color = 'blue', label = 'Training data')
plt.plot(train['sqft_living'], predictions/1000000,'-', color = 'orange', label = 'Predictions')
plt.ylabel('Price (Millions USD$)')
plt.xlabel('House Size (Square Feet)')
xl = plt.xlim()
yl = plt.ylim()
plt.legend()
plt.show()

Now, use the calculated slope and intercept to do the prediction of a single size value:

In [ ]:
size_to_predict = 15000 # sqft_living to be predicted

# YOUR CODE HERE #
prediction = 

print('Size (sqft): %.i' % size_to_predict)
print('Price (USD$): %.i' % prediction)

### Linear Regression with Multiple Features

The first thing to look at is, instead of an analytical solution, we will need to find an approximate solution.

The features can be represented by a set of arrays $x_{i}[j]$, where $j$ is the feature and $i$ is a feature element. The "linear" equation (now a multi-dimensional plane) will have the intercept and a coefficient for each feature, and is written as:

$$y_{i}^{,}(x_i) = w_0*x_{i}[0] + w_{1}*x_{i}[1] + w_{2}*x_{i}[2] + \dots + w_{d}*x_{i}[d] = \sum_{j=0}^{d}w_j*x_{i}[j]$$

where $x_i[0]=1$. We can use matrix notation and use the dot product of the parameters $\textbf{w}$ with the features $\textbf{x}_i$:

$$y_{i}^{,}(\textbf{x}_i) = \textbf{w}^{T}\textbf{x}_i$$

As for the linear regression with a single feature, the solution for multiple features come from the minimization of the ***Residual Sum of Squares*** (RSS):

$$RSS(\textbf{w}) = \sum_{i=1}^{N}(y_i - y_{i}^{,})^2 = \sum_{i=1}^{N}(y_i - \textbf{w}^{T}\textbf{x}_i)^2 = (\textbf{y} - \textbf{w}^{T}\textbf{x})^{T}(\textbf{y} - \textbf{w}^{T}\textbf{x})$$

The gradient $\Phi$ (the derivative of RSS relative to $\textbf{w}$) is:

$$\Phi = -2\textbf{x}^{T}(\textbf{y} - \textbf{w}^{T}\textbf{x})$$

In others words, we just need to take the dot product between the features and the residuals (the difference between the predictions and the measured data) and multiply by 2. The gradient is used to update the parameters $\textbf{w}$ iteratively in a way to minimize the errors (set the gradient to zero). In the [gradient descent](https://en.wikipedia.org/wiki/Gradient_descent) method, for the n-th iteration, we end up with the following equation:

$$\textbf{w}_{n+1} = \textbf{w}_{n} - \alpha\Phi = \textbf{w}_{n} + 2{\alpha}\textbf{x}^{T}(\textbf{y} - \textbf{w}^{T}_{n}\textbf{x})$$

where $\alpha$ is the learning rate, a scalar that helps the minimization to converge faster to the minimum point. The choice of the step length shows to be trick, as a large value wil lead to minimization to diverge as a too small value will diverge slowly.

Now we shall be able to compute the best set of parameters given a set of features and the output target. But to compute this, first we need to convert the data in the sframe format (the Turicreate dataframe) to a numpy numerical matrix. We will create a series of functions to combine in a final one that will calculate the parameters.

To facilitate the operations, let's convert the columns of the dataframe to numpy arrays:

In [ ]:
def get_numpy_data(df, features, output):
    # First, create a column with the feature x_0 = 1
    # YOUR CODE HERE
    df['constant'] = 

    # Include the w_0 in the features list in the first column
    features = ['constant'] + features 
    
    # From the data, pick only desired features and convert to numpy array
    # YOUR CODE HERE
    feature_matrix = 
    
    # Doing the same for the output
    # YOUR CODE HERE
    output_array = 
    
    # Return the features and output target as numerical matrices
    return(feature_matrix, output_array)

Let's test the function:

In [ ]:
(example_features, example_output) = get_numpy_data(data, ['sqft_living'], 'price') # the [] around 'sqft_living' makes it a list
print(example_features[0,:])
print(example_output[0])

Now, to make our life easier, we can use the fact that the predictions are simply the **dot product** of the **weights** with the **features**, to create a function to compute the prediction given the matrix of the features and the array of parameters (weights). For that, we will the *Numpy* function [dot](https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html).

In [ ]:
def predict_output(feature_matrix, weights):
    # The predictions are the dot product of the features and weights (parameters)
    # YOUR CODE HERE
    predictions = 
    
    # Return the predictions
    return(predictions)

Testing...

In [ ]:
# First computing manually
my_weights = np.array([1., 1.]) # the example weights
my_features = example_features[0,] # we'll use the first data point
predicted_value = np.dot(my_features, my_weights)

# Now using our function
test_predictions = predict_output(example_features, my_weights)

print(predicted_value)
print(test_predictions[0])
print(test_predictions[1])

Now, let's create a function that, by giving the residuals and features, it returns the gradient $\Phi$ (the derivative of RSS relative to $\textbf{w}$):

$$\Phi = -2\textbf{x}^{T}(\textbf{y} - \textbf{w}^{T}\textbf{x})$$

In [ ]:
def feature_derivative(residuals, feature):
    # Using the equation of the derivative, do the dot product 
    # between the features and residuals and multiply by 2
    # YOUR CODE HERE
    derivative = 
    
    # Return the derivative
    return(derivative)

Another testing...

In [ ]:
# Using only on feature
(example_features, example_output) = get_numpy_data(data, ['sqft_living'], 'price') 

# Making the parameters = 0 so the derivative should be
# equal the sum of the output. This is an easy way to
# evaluate our function
my_weights = np.array([0., 0.])

test_predictions = predict_output(example_features, my_weights) 

residuals = test_predictions - example_output

# Let's compute the derivative with respect to 'constant',
# the ":" indicates "all rows"
features = example_features[:,0] 

derivative = feature_derivative(residuals, features)

print(derivative)
print(-np.sum(example_output)*2) # should be the same as derivative
print(derivative - (-np.sum(example_output)*2)) # should be ZERO

Perfect. Now, let's head to the to the gradient descent method. 

First, we need to define a stoping criteria (tolerace), a value that stops the loop if the gradient is smaller than it. Also, it needs the initial parameters (weights) guess, and the step length (set as constant for all the iterations) as inputs.

In [ ]:
from math import sqrt

In [ ]:
def regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance):
    converged = False 
    weights = np.array(initial_weights) # make sure it's a numpy array
    iteration = 0
    
    while not converged:
        # Compitung predictions with initially current weights, then the updated ones
        predictions = predict_output(feature_matrix, weights) 
        
        # Computing current residuals
        residuals = predictions-output
        gradient_sum_squares = 0 # initialize the gradient sum of squares
        
        # while we haven't reached the tolerance yet, update each feature's weight
        for i in range(len(weights)): # loop over each weight
            # Estimating the derivative for each parameter (weight). We created a function for it
            # YOUR CODE HERE
            derivative = 
            
            # add the squared value of the derivative to the gradient magnitude (for assessing convergence)
            # YOUR CODE HERE
            gradient_sum_squares = 
            
            # subtract the step size times the derivative from the current weight
            # YOUR CODE HERE
            weights[i] =
        
        # compute the square-root of the gradient sum of squares to get the gradient magnitude:
        gradient_magnitude = sqrt(gradient_sum_squares)
        iteration = iteration + 1

        # The tolerance is our stoping criteria
        if gradient_magnitude < tolerance:
            converged = True
    
    # Return the parameters (weights) and number of iterations
    return(weights,iteration)

Testing:

In [ ]:
# Let's run the function over the train data
model_features = ['sqft_living', 'sqft_living15'] # sqft_living15 is the average squarefeet for the nearest 15 neighbors.
my_output = 'price'
feature_matrix, output = get_numpy_data(train, model_features, my_output)
initial_weights = np.array([-100000., 1., 1.])
step_size = 4e-12
tolerance = 1e9

# Gradient descent
(weights,iteration) = regression_gradient_descent(feature_matrix, output, initial_weights, step_size, tolerance)

# Predictions over the test data
(test_feature_matrix, test_output_new) = get_numpy_data(val, model_features, my_output)
prediction = predict_output(test_feature_matrix, weights)

# Checking outputs
print(weights)
print(iteration)
print(prediction[0])
print(test_output_new[0])
print(prediction[0] - test_output_new[0])
print(round(((prediction[0] - test_output_new[0])/test_output_new[0])*100,2))